<a href="https://www.kaggle.com/code/nguynvnln22028281/train-databirdclef?scriptVersionId=302646482" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm # Or standard tqdm if not in notebook
import time
import random
import cv2


import torch
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T # For SpecAugment

In [ ]:
CSV_PATH = "/kaggle/input/birdclef-2025/train.csv"
AUDIO_BASE_PATH = "/kaggle/input/birdclef-2025/train_audio/"
# Audio Processing Parameters
SAMPLE_RATE = 32000
DURATION = 5
N_MELS = 128 # EfficientNet works well with larger image sizes, 128 or 224 are common
FMIN = 20
FMAX = 16000
HOP_LENGTH = 512
N_FFT = 2048
TARGET_SHAPE = 224


VALIDATION_SPLIT = 0.2
RANDOM_SEED = 42
NUM_WORKERS = 2 # os.cpu_count()
LABEL_SMOOTHING = 0.1 # Set to 0 to disable
PRELOAD_DATA_IN_RAM = True # Set to True if you have enough RAM and want faster epochs after initial load

# --- Set Seed for Reproducibility ---
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)

# --- Determine Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Load the training metadata ---
try:
    data = pd.read_csv(CSV_PATH)
    print(f"Successfully loaded {CSV_PATH}")
    print(f"Data shape: {data.shape}")
except FileNotFoundError:
    print(f"Error: Could not find {CSV_PATH}")
    exit()

# Construct full audio file paths
data['full_path'] = data['filename'].apply(lambda x: os.path.join(AUDIO_BASE_PATH, x))


In [ ]:
# --- 2. Label Encoding ---
encoder = LabelEncoder()
data['label_encoded'] = encoder.fit_transform(data['primary_label'])
NUM_CLASSES = len(encoder.classes_)
print(f"\nNumber of unique classes: {NUM_CLASSES}")

In [ ]:
# --- 3. Train/Validation Split ---
train_df, val_df = train_test_split(
    data,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED,
    stratify=data['label_encoded'] # Crucial for imbalanced datasets
)
print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")

# Calculate expected spectrogram shape (for reference and padding)
# For librosa.stft, frame length is N_FFT. Number of frames is roughly len(y) / hop_length.
# For melspectrogram, the time dimension is ceil(samples / hop_length)
# After padding/truncating audio to DURATION * SAMPLE_RATE:
TARGET_LENGTH_SAMPLES = int(SAMPLE_RATE * DURATION)
N_FRAMES = int(np.ceil(TARGET_LENGTH_SAMPLES / HOP_LENGTH)) + 1 # Librosa can add a frame due to centering
print(f"Expected Spectrogram Shape (H, W) after processing: ({N_MELS}, {N_FRAMES})")

In [ ]:
# --- 4. Audio Preprocessing & Augmentation Function ---
def load_and_preprocess_audio(
    file_path, target_sr=SAMPLE_RATE, duration_samples=TARGET_LENGTH_SAMPLES,
    n_mels=N_MELS, hop_length=HOP_LENGTH, n_fft=N_FFT,
    fmin=FMIN, fmax=FMAX, is_train=False):
    """Loads, augments (if train), pads/truncates, and computes Mel spectrogram."""
    try:
        y, sr = librosa.load(file_path, sr=None) # Load native SR
        if sr != target_sr:
            y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

        # Pad or truncate audio to target_length_samples
        if len(y) < duration_samples:
            padding = duration_samples - len(y)
            offset = random.randint(0, padding) if is_train else padding // 2 # Pad randomly for train
            y = np.pad(y, (offset, padding - offset), mode='constant')
        elif len(y) > duration_samples:
            start = random.randint(0, len(y) - duration_samples) if is_train else 0 # Random crop for train
            y = y[start : start + duration_samples]

        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=target_sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, fmin=fmin, fmax=fmax, window='hann' # Hann window is common
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalize to [0, 1] (or standardize if preferred)
        min_val = np.min(mel_spec_db)
        max_val = np.max(mel_spec_db)
        if max_val > min_val:
            mel_spec_db = (mel_spec_db - min_val) / (max_val - min_val)
        else: # Handle silent clips
            mel_spec_db = np.zeros_like(mel_spec_db)

        # Ensure consistent shape (especially time dimension) due to potential minor librosa variations
        if mel_spec_db.shape[1] < N_FRAMES:
            pad_width = N_FRAMES - mel_spec_db.shape[1]
            mel_spec_db = np.pad(mel_spec_db, ((0,0), (0, pad_width)), mode='constant', constant_values=0.)
        elif mel_spec_db.shape[1] > N_FRAMES:
            mel_spec_db = mel_spec_db[:, :N_FRAMES]

        mel_spec_resized = cv2.resize(mel_spec_db, (TARGET_SHAPE, TARGET_SHAPE), interpolation=cv2.INTER_LINEAR)
        return mel_spec_resized
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return np.zeros((n_mels, N_FRAMES)) # Return zeros on error, with expected shape

In [ ]:
# --- 5. Create PyTorch Dataset ---+
class BirdSoundDataset(Dataset):
    def __init__(self, dataframe, is_train=False, preload_in_ram=PRELOAD_DATA_IN_RAM,
                 apply_spec_augment=True):
        self.dataframe = dataframe
        self.is_train = is_train
        self.preload_in_ram = preload_in_ram
        self.apply_spec_augment = apply_spec_augment

        if self.is_train and self.apply_spec_augment:
            # SpecAugment: Applied on the Mel spectrogram tensor
            self.spec_augmenter =torch.nn.Sequential(
                    T.FrequencyMasking(freq_mask_param=N_MELS // 8),
                    T.TimeMasking(time_mask_param=int(N_FRAMES * 0.1))
                )
            # self.spec_augmenter = nn.Sequential( # Alternative definition
            #     T.FrequencyMasking(freq_mask_param=N_MELS // 8), # Mask up to 1/8th of mel bands
            #     T.TimeMasking(time_mask_param=int(N_FRAMES * 0.1)) # Mask up to 10% of time steps
            # )
        else:
            self.spec_augmenter = None

        if self.preload_in_ram:
            self.spectrograms = []
            self.labels = []
            print(f"Preloading {len(dataframe)} audio files into RAM for {'training' if is_train else 'validation'}...")
            for idx in tqdm(range(len(dataframe))):
                row = self.dataframe.iloc[idx]
                file_path = row['full_path']
                label = row['label_encoded']
                # For preloading, we don't apply instance-specific augmentations like SpecAugment here,
                # as they should be random for each epoch. Audio-level augs are applied during loading.
                spectrogram = load_and_preprocess_audio(file_path, is_train=self.is_train) # is_train for audio augs
                spectrogram_tensor = torch.tensor(spectrogram, dtype=torch.float32).unsqueeze(0)
                spectrogram_tensor_3channel = spectrogram_tensor.repeat(3, 1, 1)
                self.spectrograms.append(spectrogram_tensor_3channel)
                self.labels.append(torch.tensor(label, dtype=torch.long))
            print(f"All {len(self.spectrograms)} spectrograms loaded into memory!")

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        if self.preload_in_ram:
            spectrogram_3channel = self.spectrograms[idx]
            label = self.labels[idx]
        else:
            row = self.dataframe.iloc[idx]
            file_path = row['full_path']
            label_val = row['label_encoded']

            # Load and process audio (includes audio-level augmentations if is_train)
            spectrogram = load_and_preprocess_audio(file_path, is_train=self.is_train)
            spectrogram_tensor = torch.tensor(spectrogram, dtype=torch.float32).unsqueeze(0) # (1, H, W)
            spectrogram_3channel = spectrogram_tensor.repeat(3, 1, 1) # (3, H, W)
            label = torch.tensor(label_val, dtype=torch.long)

        # Apply SpecAugment (if training and enabled)
        if self.is_train and self.spec_augmenter and random.random() < APPLY_AUGMENTATION_PROB:
            # Ensure spec_augmenter is on the same device if it contains parameters,
            # or apply before moving tensor to GPU in training loop.
            # For torchaudio.transforms, they are typically stateless or handle device internally.
            try:
                spectrogram_3channel = self.spec_augmenter(spectrogram_3channel)
            except Exception as e: # Catch potential errors with SpecAugment on edge cases
                # print(f"SpecAugment error for sample {idx}, shape {spectrogram_3channel.shape}: {e}")
                pass # Skip SpecAugment for this sample if it errors

        return spectrogram_3channel, label

train_data = BirdSoundDataset(train_df, is_train=True, preload_in_ram=PRELOAD_DATA_IN_RAM)

In [ ]:
torch.save(train_data, "/kaggle/working/train_dataset.pt")

In [ ]:
val_data = BirdSoundDataset(val_df, is_train=False, preload_in_ram=PRELOAD_DATA_IN_RAM, apply_spec_augment=False)

In [ ]:
torch.save(val_data, "/kaggle/working/val_dataset.pt")